In [ ]:

def build_densenet_model(num_classes):
    # 1. Grayscale Input
    img_input = layers.Input(shape=(128, 128, 1))
    
    # 2. Preprocessing: Scale [0, 255] to [-1, 1]
    # DenseNet121 typically expects the same normalization as MobileNet
    x = layers.Rescaling(1./127.5, offset=-1)(img_input)
    
    # 3. Adapter: 1-to-3 Channel Expansion
    x = layers.Conv2D(3, (1, 1), padding='same')(x)
    
    # 4. Base Model
    base_model = tf.keras.applications.DenseNet121(
        input_shape=(128, 128, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # 5. Connect and build Head
    x = base_model(x, training=False) 
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(img_input, outputs)
    
    return model, base_model